# 爆速・常に綺麗・位置ズレなしの血管中心線抽出スクリプト

重い3Dスケルトン化を高速アルゴリズムに置き換え、`skan`ライブラリを使って血管の走行順に完璧に座標をソートします。出力形式は元のピッタリ位置が合う仕様を完全維持しています。

In [3]:
# Cell 1: 必要ライブラリのインポート
import os
import json
import numpy as np
import SimpleITK as sitk
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize

# skanからはSkeletonだけを読み込む（最もエラーが起きない安全な方法）
from skan import Skeleton

In [4]:
# Cell 2: DICOMデータの読み込み（位置データ完全同期）
DICOM_DIR = '/Users/tkimura/Desktop/Image data/LTxD/5089291/DICOM/DICOM-A/ScalarVolume_76'

reader = sitk.ImageSeriesReader()
dicom_names = reader.GetGDCMSeriesFileNames(DICOM_DIR)
reader.SetFileNames(dicom_names)
image = reader.Execute()

data_np = sitk.GetArrayFromImage(image)
spacing = np.array(image.GetSpacing())
origin = np.array(image.GetOrigin())

print(f"Volume Shape: {data_np.shape}")
print(f"Spacing (mm): {spacing}")
print(f"Origin (mm): {origin}")

Volume Shape: (158, 512, 512)
Spacing (mm): [0.743 0.743 2.   ]
Origin (mm): [ -190.057  -190.058 -1245.5  ]


In [5]:
# Cell 3: 領域抽出（二値化）
# 最も末梢が綺麗に抜けていた閾値を指定してください（例: min=150, max=1000）
th_min = 150
th_max = 1000

binary_vessel = (data_np >= th_min) & (data_np <= th_max)
binary_vessel = binary_vessel.astype(np.uint8)
print("Segmentation complete.")

Segmentation complete.


In [6]:
# Cell 4: 爆速スケルトン化 ＆ 距離マップ計算
print("Processing Skeletonization (Optimized Method)...")
dist_map = distance_transform_edt(binary_vessel)

# 【改善点】skeletonize_3d から 高速な Leeの2D/3Dハイブリッドアルゴリズムに変更
skeleton = skeletonize(binary_vessel)
print("Skeletonization complete.")

Processing Skeletonization (Optimized Method)...
Skeletonization complete.


In [12]:
# Cell 5: skan を用いた中心線抽出（完全バージョン非依存・疎行列対策版）
print("Sorting and refining vessel paths...")

# スケルトンオブジェクトの作成
skel_obj = Skeleton(skeleton)

all_segments = []

# 【修正点】skel_obj.paths が SciPy の疎行列であることを考慮し、
# 行列の「行数（shape[0]）」を取得して安全に全エッジをループさせます
num_edges = skel_obj.paths.shape[0]

for edge_idx in range(num_edges):
    
    # スケルトンのトポロジーを解析し、端から順に綺麗に並んだピクセル座標 (z, y, x) を取得
    path_pixels = skel_obj.path_coordinates(edge_idx)
    
    # ノイズ除去：5画素未満のぶつ切りデータは無視
    if len(path_pixels) < 5:
        continue
        
    # (x, y, z) の順に並び替えて格納
    pts = path_pixels[:, [2, 1, 0]].tolist()
    all_segments.append(pts)

print(f"Extracted {len(all_segments)} perfectly ordered vessel segments.")

Sorting and refining vessel paths...
Extracted 9876 perfectly ordered vessel segments.


In [13]:
# Cell 6: 元のスクリプトと100%互換のある完璧な座標で JSON 保存
vol_center_idx = np.array(data_np.shape[::-1]) / 2.0
vol_center_mm = origin + vol_center_idx * spacing

output_data = {
    "metadata": {"unit": "mm", "vessel_count": len(all_segments)},
    "polylines": []
}

for idx, seg in enumerate(all_segments):
    blender_nodes = []
    unity_nodes = []
    
    for pt in seg:
        pt_np = np.array(pt)
        phys_mm = origin + pt_np * spacing
        centered_mm = phys_mm - vol_center_mm
        
        radius_vox = dist_map[int(pt[2]), int(pt[1]), int(pt[0])]
        radius_mm = radius_vox * spacing[0]
        
        blender_nodes.append({
            "pos": centered_mm.tolist(),
            "r": float(radius_mm)
        })
        
        unity_nodes.append({
            "pos": [centered_mm[0], centered_mm[2], centered_mm[1]],
            "r": float(radius_mm)
        })
        
    output_data["polylines"].append({
        "id": idx,
        "blender": blender_nodes,
        "unity": unity_nodes
    })

output_path = "vessel_centerline_dual.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

print(f"[SUCCESS] Saved perfect JSON to: {os.path.abspath(output_path)}")

[SUCCESS] Saved perfect JSON to: /Users/tkimura/Desktop/vessel_centerline_dual.json
